<a href="https://colab.research.google.com/github/DevHuney/tg-cloner/blob/main/tg-cloner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
TG Cloner - Versão Colab
GitHub.com/DevHuney/tg-cloner
"""

!pip install telethon

import asyncio
import random
import os
import json
import locale
import shutil
from google.colab import drive
from telethon import TelegramClient, types
from telethon.sessions import StringSession
from PIL import Image

# ================== CONFIGURAÇÃO DO COLAB ==================
drive.mount('/content/drive')
CONFIG_PATH = '/content/drive/MyDrive/tg_cloner_config.json'
TEMP_DIR = "/content/tg_temp"
os.makedirs(TEMP_DIR, exist_ok=True)

BANNER = """
\033[1;36m
████████╗ ██████╗      ██████╗██╗     ██████╗ ███╗   ██╗███████╗██████╗
╚══██╔══╝██╔════╝     ██╔════╝██║    ██╔═══██╗████╗  ██║██╔════╝██╔══██╗
   ██║   ██║  ███╗    ██║     ██║    ██║   ██║██╔██╗ ██║█████╗  ██████╔╝
   ██║   ██║   ██║    ██║     ██║    ██║   ██║██║╚██╗██║██╔══╝  ██╔══██╗
   ██║   ╚██████╔╝    ╚██████╗███████╗╚██████╔╝██║ ╚████║███████╗██║  ██║
   ╚═╝    ╚═════╝      ╚═════╝╚══════╝ ╚═════╝ ╚═╝  ╚═══╝╚══════╝╚═╝  ╚═╝
\033[1;33m                   GitHub.com/seuuser/tg-cloner\033[0m
"""

TRANSLATIONS = {
    'PT': {
        'welcome': "★ TG Cloner ★ - Iniciando clone!",
        'config_missing': "Configuração inicial:",
        'enter_api_id': "Digite o API ID: ",
        'enter_api_hash': "Digite o API HASH: ",
        'enter_phone': "Número de telefone (ex: +5511987654321): ",
        'login_code': "Digite o código recebido: ",
        '2fa_warning': "Senha 2FA necessária: ",
        'cloning_header': "\n=== Progresso do Clone ===",
        'progress': "Progresso: {} de {} mensagens",
        'media_download': "Mídia detectada | Baixando...",
        'sending_delay': "Próximo envio em {} segundos",
        'completed': "\n✅ Clone concluído!",
        'error': "❌ Erro: {}",
        'select_source': "Índice do canal de origem: ",
        'select_target': "Índice do canal de destino: ",
        'processing_text': "Processando mensagem de texto...",
        'sticker_sending': "Enviando figurinha...",
        'video_processing': "Processando vídeo (streaming)...",
        'doc_processing': "Processando documento..."
    }
}

def show_banner():
    print(BANNER)
    print("\033[1;32m" + TRANSLATIONS['PT']['welcome'] + "\033[0m\n")

def prepare_thumbnail():
    if os.path.exists('thumbnail.jpg'):
        try:
            thumb = Image.open('thumbnail.jpg')
            thumb.thumbnail((320, 320))
            thumb_path = os.path.join(TEMP_DIR, "thumb.jpg")
            thumb.save(thumb_path)
            return thumb_path
        except Exception as e:
            print(f"Erro na thumbnail: {str(e)}")
    return None

async def copy_message(client, source_chat, dest_chat, message):
    """Versão corrigida que mantém formatação original"""
    try:
        if message.action or (not message.text and not message.media):
            return

        entities = message.entities if message.entities else []
        text = message.raw_text

        if message.media:
            if isinstance(message.media, types.MessageMediaDocument):
                document = message.media.document

                # Stickers
                if any(isinstance(attr, types.DocumentAttributeSticker) for attr in document.attributes):
                    await client.send_file(dest_chat, document)
                    print(TRANSLATIONS['PT']['sticker_sending'])
                    return

                # Vídeos
                if any(isinstance(attr, types.DocumentAttributeVideo) for attr in document.attributes):
                    thumb = prepare_thumbnail()
                    file_path = await client.download_media(message.media, file=os.path.join(TEMP_DIR, f"video_{message.id}.mp4"))
                    await client.send_file(
                        dest_chat,
                        file_path,
                        caption=text,
                        supports_streaming=True,
                        thumb=thumb,
                        attributes=document.attributes
                    )
                    os.remove(file_path)
                    print(TRANSLATIONS['PT']['video_processing'])
                    return

                # Documentos
                file_path = await client.download_media(message.media)
                await client.send_file(dest_chat, file_path, caption=text)
                os.remove(file_path)
                print(TRANSLATIONS['PT']['doc_processing'])
                return

            # Outras mídias
            file_path = await client.download_media(message.media)
            await client.send_file(dest_chat, file_path, caption=text)
            os.remove(file_path)
            print(TRANSLATIONS['PT']['media_download'])

        else:
            await client.send_message(
                dest_chat,
                message=text,
                formatting_entities=entities,
                parse_mode=None,
                link_preview=False
            )
            print(TRANSLATIONS['PT']['processing_text'])

        delay = random.randint(5, 15)
        print(TRANSLATIONS['PT']['sending_delay'].format(delay))
        await asyncio.sleep(delay)

    except Exception as e:
        print(f"\n\033[1;31m{TRANSLATIONS['PT']['error'].format(str(e))}\033[0m")

async def main():
    show_banner()
    client = None
    try:
        if os.path.exists(CONFIG_PATH):
            with open(CONFIG_PATH) as f:
                config = json.load(f)

            client = TelegramClient(
                StringSession(config['session']),
                int(config['api_id']),
                config['api_hash']
            )
            await client.start()
        else:
            print(f"\033[1;33m{TRANSLATIONS['PT']['config_missing']}\033[0m")
            api_id = input(f"\033[1;34m{TRANSLATIONS['PT']['enter_api_id']}\033[0m")
            api_hash = input(f"\033[1;34m{TRANSLATIONS['PT']['enter_api_hash']}\033[0m")
            phone = input(f"\033[1;34m{TRANSLATIONS['PT']['enter_phone']}\033[0m")

            client = TelegramClient(StringSession(), int(api_id), api_hash)
            await client.start(
                phone,
                code_callback=lambda: input(f"\033[1;34m{TRANSLATIONS['PT']['login_code']}\033[0m"),
                password=lambda: input(f"\033[1;34m{TRANSLATIONS['PT']['2fa_warning']}\033[0m")
            )

            with open(CONFIG_PATH, 'w') as f:
                json.dump({
                    'api_id': api_id,
                    'api_hash': api_hash,
                    'session': client.session.save()
                }, f)

        async with client:
            # Listar canais
            dialogs = await client.get_dialogs()
            channels = [d for d in dialogs if d.is_channel or d.is_group]

            print("\n\033[1;34mCanais disponíveis:\033[0m")
            for i, c in enumerate(channels):
                print(f"[{i}] {c.name}")

            # Selecionar canais
            origem_idx = int(input(f"\n\033[1;36m{TRANSLATIONS['PT']['select_source']}\033[0m"))
            destino_idx = int(input(f"\033[1;36m{TRANSLATIONS['PT']['select_target']}\033[0m"))
            origem = channels[origem_idx]
            destino = channels[destino_idx]

            # Coletar mensagens
            messages = []
            async for message in client.iter_messages(origem, reverse=True):
                messages.append(message)

            # Processar mensagens
            total = len(messages)
            print(f"\n\033[1;35m» Iniciando clone de {total} mensagens\033[0m")

            for idx, msg in enumerate(messages, 1):
                print(f"\r\033[1;34m[{idx}/{total}]\033[0m", end='', flush=True)
                await copy_message(client, origem.id, destino.id, msg)

            print(f"\n\n\033[1;32m{TRANSLATIONS['PT']['completed']}\033[0m")

    except Exception as e:
        print(f"\n\033[1;31m{TRANSLATIONS['PT']['error'].format(str(e))}\033[0m")
    finally:
        if client:
            await client.disconnect()
        shutil.rmtree(TEMP_DIR, ignore_errors=True)

if __name__ == '__main__':
    asyncio.run(main())